# Notebook: nb_bronze_customers
# Purpose: Ingest customers into bronze lakehouse
# Layer: Bronze (raw ingestion)
# Source: csv - Files/raw/customers/*.csv
# Target: bronze_customers

In [ ]:
# --- Parameters ---
source_name = "customers"
source_format = "csv"  # csv | parquet | json
source_path = "Files/raw/customers/*.csv"
load_mode = "append"  # append | overwrite (use append for bronze)

In [ ]:
# --- Imports ---
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [ ]:
# --- Read Source Data ---
df_raw = spark.read.format(source_format) \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(source_path)

print(f"Source rows: {df_raw.count()}")
print(f"Source columns: {df_raw.columns}")

In [ ]:
# --- Add Metadata Columns ---
df_bronze = df_raw \
    .withColumn("_load_timestamp", F.current_timestamp()) \
    .withColumn("_source_file", F.input_file_name()) \
    .withColumn("_load_id", F.lit(
        notebookutils.runtime.context.get("currentRunId", "manual")
    ))

In [ ]:
# --- Write to Delta Table ---
df_bronze.write.format("delta") \
    .mode(load_mode) \
    .option("mergeSchema", "true") \
    .saveAsTable(f"bronze_{source_name}")

print(f"Written to: bronze_{source_name}")

In [ ]:
# --- Validation ---
rows_written = spark.table(f"bronze_{source_name}").count()
print(f"Source rows: {df_raw.count()}")
print(f"Table total rows: {rows_written}")

assert rows_written > 0, f"FAIL: No rows in bronze_{source_name}"
print("PASS: Bronze load complete")